# Export DNN NPZ Results to CSV

This notebook reads all DNN `.npz` result files from `results` (or `resutls`, if that is the folder name on the other computer) and exports a 60-row CSV. The exported R2 values are the 50th saved train/test R2 entries, not the last entries.

In [ ]:
from pathlib import Path
import csv
import re
import numpy as np

# Notebook is intended to live in all_models/dnn.
DNN_DIR = Path.cwd()
if DNN_DIR.name != "dnn" and (DNN_DIR / "all_models" / "dnn").exists():
    DNN_DIR = DNN_DIR / "all_models" / "dnn"

RESULTS_DIR_CANDIDATES = [DNN_DIR / "results", DNN_DIR / "resutls"]
RESULTS_DIR = next(
    (path for path in RESULTS_DIR_CANDIDATES if path.exists() and any(path.glob("*.npz"))),
    next((path for path in RESULTS_DIR_CANDIDATES if path.exists()), RESULTS_DIR_CANDIDATES[0]),
)
OUTPUT_CSV = RESULTS_DIR / "dnn_results_50th_saved_r2_summary.csv"

# 50th saved value means one-based position 50, i.e. Python index 49.
SAVED_R2_POSITION = 50
SAVED_R2_INDEX = SAVED_R2_POSITION - 1

# Required row order: first r2=0.3 seeds 1-20, then 0.5, then 0.8.
R2_ORDER = [0.3, 0.5, 0.8]
SEED_ORDER = list(range(1, 21))

print(f"DNN directory: {DNN_DIR}")
print(f"Reading NPZ files from: {RESULTS_DIR}")
print(f"Writing CSV to: {OUTPUT_CSV}")

In [ ]:
def normalize_scalar(value):
    """Convert numpy scalar/small-array values to CSV-friendly Python values."""
    array = np.asarray(value)
    if array.shape == ():
        item = array.item()
        return item.item() if hasattr(item, "item") else item
    if array.size == 1:
        item = array.reshape(-1)[0]
        return item.item() if hasattr(item, "item") else item
    return "[" + ", ".join(str(x.item() if hasattr(x, "item") else x) for x in array.reshape(-1)) + "]"


def parse_float_token(token):
    return float(token.replace("m", "-").replace("p", "."))


def metadata_from_filename(path):
    pattern = re.compile(
        r"dnn_results_r2_(?P<r2>[^_]+)_epochs_(?P<epochs>\d+)_seed_(?P<seed>\d+)_lr_(?P<lr>[^.]+)\.npz$"
    )
    match = pattern.match(path.name)
    if not match:
        return {}
    return {
        "r2": parse_float_token(match.group("r2")),
        "epochs": int(match.group("epochs")),
        "seed": int(match.group("seed")),
        "lr": parse_float_token(match.group("lr")),
    }


def get_50th_saved_r2(bundle, key, path):
    values = np.asarray(bundle[key], dtype=float).reshape(-1)
    if values.size <= SAVED_R2_INDEX:
        raise ValueError(
            f"{path.name} has only {values.size} saved values for {key}; "
            f"need at least {SAVED_R2_POSITION}."
        )
    return float(values[SAVED_R2_INDEX])


def load_result_row(path):
    with np.load(path, allow_pickle=True) as bundle:
        row = metadata_from_filename(path)

        # Prefer metadata saved inside the bundle. Keep all scalar/small parameter settings.
        metric_prefixes = ("r2_", "loss_")
        for key in bundle.files:
            if key.startswith(metric_prefixes):
                continue
            value = bundle[key]
            if np.asarray(value).size <= 20:
                column = "r2" if key == "true_r2" else key
                row[column] = normalize_scalar(value)

        if "r2" not in row and "true_r2" in row:
            row["r2"] = row.pop("true_r2")

        row["r2_train_50th_saved"] = get_50th_saved_r2(bundle, "r2_train", path)
        row["r2_test_50th_saved"] = get_50th_saved_r2(bundle, "r2_test", path)
        row["saved_r2_position"] = SAVED_R2_POSITION
        row["source_file"] = path.name

    row["r2"] = float(row["r2"])
    row["seed"] = int(row["seed"])
    return row


def sort_key(row):
    rounded_r2 = round(float(row["r2"]), 10)
    try:
        r2_rank = [round(x, 10) for x in R2_ORDER].index(rounded_r2)
    except ValueError:
        r2_rank = len(R2_ORDER)
    return (r2_rank, int(row["seed"]))

In [ ]:
npz_files = sorted(RESULTS_DIR.glob("*.npz"))
if not npz_files:
    raise FileNotFoundError(f"No .npz files found in {RESULTS_DIR}")

rows = [load_result_row(path) for path in npz_files]
rows = sorted(rows, key=sort_key)

expected_pairs = [(r2, seed) for r2 in R2_ORDER for seed in SEED_ORDER]
actual_pairs = [(round(float(row["r2"]), 10), int(row["seed"])) for row in rows]
expected_pairs_rounded = [(round(r2, 10), seed) for r2, seed in expected_pairs]

missing_pairs = [pair for pair in expected_pairs_rounded if pair not in actual_pairs]
extra_pairs = [pair for pair in actual_pairs if pair not in expected_pairs_rounded]

if len(rows) != 60:
    raise ValueError(f"Expected 60 rows, found {len(rows)} rows.")
if missing_pairs:
    raise ValueError(f"Missing expected (r2, seed) pairs: {missing_pairs}")
if extra_pairs:
    raise ValueError(f"Found unexpected (r2, seed) pairs: {extra_pairs}")

# Put the important settings first, then any extra saved scalar settings, then metrics/source.
preferred_columns = [
    "r2",
    "seed",
    "epochs",
    "sample_size",
    "val_ratio",
    "in_feature_list",
    "lr",
    "saved_r2_position",
    "r2_train_50th_saved",
    "r2_test_50th_saved",
    "source_file",
]
all_columns = []
for row in rows:
    for column in row:
        if column not in all_columns:
            all_columns.append(column)
fieldnames = [column for column in preferred_columns if column in all_columns]
fieldnames += [column for column in all_columns if column not in fieldnames]

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_CSV.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

print(f"Exported {len(rows)} rows to {OUTPUT_CSV}")
print("First 5 exported rows:")
for row in rows[:5]:
    print({key: row.get(key) for key in fieldnames})

In [ ]:
# Optional: display the exported CSV in a table if pandas is installed.
try:
    import pandas as pd
    display(pd.read_csv(OUTPUT_CSV))
except ImportError:
    print("pandas is not installed; CSV export is complete.")